The **$\text{SU}(2)$ matrix** represents a unitary beam splitter transformation operating on two optical modes, defined by a transmission/reflection mixing angle $\theta$ and a phase shift $\phi$. Because it belongs to the Special Unitary group of degree 2, it satisfies $U^\dagger U = I$ and $\det(U) = 1$, ensuring total photon probability and energy are perfectly conserved. In Linear Optical Quantum Computing (LOQC), these $2 \times 2$ transformations serve as fundamental building blocks: by embedding them into a larger $6 \times 6$ identity matrix via embed_SU2, single-photon creations across independent spatial modes are unitarily combined to implement complex 2-qubit logic operations like the CNOT gate.

In [ ]:
import numpy as np
import strawberryfields as sf
from strawberryfields.ops import Fock, BSgate

def SU2(theta, phi):
    """
        U(theta, phi) =[[ cos(theta),            exp(+iphi)sin(theta)], 
                        [-exp(-iphi)sin(theta),  cos(theta)]]
    """
    return np.array([[np.cos(theta), np.exp(1j*phi)*np.sin(theta)], 
                     [-np.exp(-1j*phi)*np.sin(theta), np.cos(theta)]], dtype=complex)


def embed_SU2(U, mode_a, mode_b, n_modes=6):
    M = np.eye(n_modes, dtype=complex) # creates 6x6 I matrix
    indices = [mode_a, mode_b] # the modes to embed the SU(n) matrix into
    M[np.ix_(indices, indices)] = U # embed the SU(n) matrix into the identity matrix
    return M

In [ ]:
theta1 = np.arccos(1/np.sqrt(3))
theta2 = np.pi/4
theta3 = np.arccos(1/np.sqrt(3))
theta4 = np.arccos(1/np.sqrt(3))
theta5 = np.pi/4

phi1 = 0.0
phi2 = 0.0
phi3 = 0.0
phi4 = 0.0
phi5 = np.pi

U1 = SU2(theta1, phi1)
U2 = SU2(theta2, phi2)
U3 = SU2(theta3, phi3)
U4 = SU2(theta4, phi4)
U5 = SU2(theta5, phi5)

print("SU(2) Matrices\n")
for i, U in enumerate([U1, U2, U3, U4, U5], start=1):
    print(f"U{i} =")
    print(U, 6) 

SU(2) Matrices

U1 =
[[ 0.57735027+0.j  0.81649658+0.j]
 [-0.81649658+0.j  0.57735027+0.j]] 6
U2 =
[[ 0.70710678+0.j  0.70710678+0.j]
 [-0.70710678+0.j  0.70710678+0.j]] 6
U3 =
[[ 0.57735027+0.j  0.81649658+0.j]
 [-0.81649658+0.j  0.57735027+0.j]] 6
U4 =
[[ 0.57735027+0.j  0.81649658+0.j]
 [-0.81649658+0.j  0.57735027+0.j]] 6
U5 =
[[ 0.70710678+0.00000000e+00j -0.70710678+8.65956056e-17j]
 [ 0.70710678+8.65956056e-17j  0.70710678+0.00000000e+00j]] 6


In [67]:
print("Check if unitary: ")
for i, U in enumerate([U1, U2, U3, U4, U5], start=1):
    unitary = np.allclose(U.conj().T@U, np.eye(2))  # compares  Udagger@U with 2x2 I matrix
    determinant = np.linalg.det(U)
    print(f"U{i}: ", {unitary}, f"det(U) = {determinant}")

M1 = embed_SU2(U1, 0, 1)
M2 = embed_SU2(U2, 3, 4)
M3 = embed_SU2(U3, 2, 3)
M4 = embed_SU2(U4, 4, 5)
M5 = embed_SU2(U5, 3, 4)
M_total = M5@M4@M3@M2@M1

print("Complete 6x6 matrix\n")
print(np.round(M_total, 6))
print("\nCheck if unitary: ")
print(np.allclose((M_total.conj().T)@M_total, np.eye(6))) # compares  M_total dagger@M_total with 6x6 I matrix

Check if unitary: 
U1:  {True} det(U) = (1+0j)
U2:  {True} det(U) = (1+0j)
U3:  {True} det(U) = (1+0j)
U4:  {True} det(U) = (1+0j)
U5:  {True} det(U) = (1+0j)
Complete 6x6 matrix

[[ 0.57735 +0.j  0.816497+0.j  0.      +0.j  0.      +0.j  0.      +0.j
   0.      +0.j]
 [-0.816497+0.j  0.57735 +0.j  0.      +0.j  0.      +0.j  0.      +0.j
   0.      +0.j]
 [ 0.      +0.j  0.      +0.j  0.57735 +0.j  0.57735 +0.j  0.57735 +0.j
   0.      +0.j]
 [ 0.      +0.j  0.      +0.j -0.57735 +0.j  0.57735 -0.j -0.      +0.j
  -0.57735 +0.j]
 [ 0.      +0.j  0.      +0.j -0.57735 -0.j -0.      +0.j  0.57735 +0.j
   0.57735 +0.j]
 [ 0.      +0.j  0.      +0.j  0.      +0.j  0.57735 +0.j -0.57735 +0.j
   0.57735 +0.j]]

Check if unitary: 
True


In [68]:
def cnot(input_state, theta1, phi1, theta2, phi2, theta3, phi3, theta4, phi4, theta5, phi5, 
                  cutoff_dim=3):

    prog = sf.Program(6)
    with prog.context as q:
        for mode, photons in enumerate(input_state):
            if photons > 0:
                Fock(photons) | q[mode]
        BSgate(theta1, phi1) | (q[0], q[1])
        BSgate(theta2, phi2) | (q[3], q[4])
        BSgate(theta3, phi3) | (q[2], q[3])
        BSgate(theta4, phi4) | (q[4], q[5])
        BSgate(theta5, phi5) | (q[3], q[4])

    eng = sf.Engine("fock", backend_options={"cutoff_dim": cutoff_dim})
    result = eng.run(prog)
    return result


In [69]:
params = [theta1, phi1, theta2, phi2, theta3, phi3, theta4, phi4, theta5, phi5]

inputs = {
    "|00>": [0, 1, 0, 1, 0, 0],
    "|01>": [0, 1, 0, 0, 1, 0],
    "|10>": [0, 0, 1, 1, 0, 0],
    "|11>": [0, 0, 1, 0, 1, 0]
}

desired_outputs = {
    "|00>": (0, 1, 0, 1, 0, 0),
    "|01>": (0, 1, 0, 0, 1, 0),
    "|10>": (0, 0, 1, 0, 1, 0),
    "|11>": (0, 0, 1, 1, 0, 0)
}

print("\nCNOT (Ralph et al)\n")

for name, input_state in inputs.items():
    result = cnot(input_state, *params)
    probs = result.state.all_fock_probs()
    desired_output = desired_outputs[name]
    success_prob = probs[desired_output]

    print(f"{name} -> desired output = {desired_output}")
    print(f"Success probability = ", {success_prob})


CNOT (Ralph et al)



c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\strawberryfields\program.py:732: UserWarning: The circuit consists of 2 disconnected components.


|00> -> desired output = (0, 1, 0, 1, 0, 0)
Success probability =  {0.11111111111111122}
|01> -> desired output = (0, 1, 0, 0, 1, 0)
Success probability =  {0.11111111111111122}
|10> -> desired output = (0, 0, 1, 0, 1, 0)
Success probability =  {0.11111111111111113}
|11> -> desired output = (0, 0, 1, 1, 0, 0)
Success probability =  {0.11111111111111113}
